# 多层感知机

In [1]:
from torch import nn
import torch.nn.functional as F

In [2]:
class NeuralNetwork(nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super(NeuralNetwork, self).__init__()

        self.layers = nn.Sequential(
            # 第1个隐藏层
            nn.Linear(num_inputs, 30),
            # nn.ReLU(),  # 激活函数
            nn.GELU(),  # 激活函数

            # 第2个隐藏层
            nn.Linear(30, 20),
            # nn.ReLU(),  # 激活函数
            nn.GELU(),  # 激活函数

            # 输出层
            nn.Linear(20, num_outputs),
        )

    def forward(self, x):
        logits = self.layers(x)  # 前向传播，得到输出层的线性输出
        return logits
    
# 说明：
# nn.Sequential() 是一个容器，可以将多个层按顺序组合在一起。
# 每个 nn.Linear() 定义了一个全连接层，nn.ReLU() 是激活函数。forward() 方法定义了前向传播的计算过程。
# nn.Linear 层会将输入和权重相乘，并加上偏置，得到线性输出。有时被称为 前馈层 或者 全连接层


In [3]:
model = NeuralNetwork(num_inputs=50, num_outputs=3)
print(model)

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): GELU(approximate='none')
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)


In [4]:
# 参数
print('预测参数量:', 50*30+30 + 30*20+20 + 20*3+3)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print('实际参数量:', num_params)

预测参数量: 2213
实际参数量: 2213


In [5]:
# 权重
print('第1个隐藏层权重:', model.layers[0].weight)
print('第2个隐藏层权重:', model.layers[2].weight)
print('输出层权重:', model.layers[4].weight)

第1个隐藏层权重: Parameter containing:
tensor([[-0.0784, -0.0485,  0.1190,  ...,  0.1341, -0.1294,  0.0158],
        [-0.0847, -0.0580,  0.0065,  ..., -0.1402, -0.0345,  0.0411],
        [ 0.1000,  0.0286,  0.0423,  ..., -0.1090,  0.0051,  0.0411],
        ...,
        [-0.0766, -0.1210,  0.0694,  ..., -0.1313, -0.0485, -0.0213],
        [-0.0365,  0.1200, -0.0930,  ..., -0.0004,  0.0267, -0.1196],
        [ 0.0749, -0.1004, -0.0559,  ...,  0.0730,  0.0525,  0.0230]],
       requires_grad=True)
第2个隐藏层权重: Parameter containing:
tensor([[ 0.0048,  0.0783, -0.0620,  0.1382, -0.1578, -0.0054,  0.1127,  0.1403,
         -0.1724,  0.0715, -0.1746, -0.0970,  0.0199, -0.0461,  0.0584,  0.0130,
         -0.0943, -0.0699, -0.1578, -0.0897,  0.1118,  0.0991, -0.1527,  0.0400,
          0.0931, -0.0357,  0.0924, -0.1645,  0.0218, -0.0080],
        [-0.0851,  0.1815,  0.1778,  0.1539,  0.0241, -0.1409,  0.1043, -0.1799,
          0.1544, -0.1145, -0.1425, -0.1175,  0.1825,  0.0993, -0.0414, -0.1323,
      

# 模型预测

In [11]:
import torch
model = NeuralNetwork(num_inputs=50, num_outputs=3)

torch.manual_seed(123)  # 设置随机种子，确保每次运行结果相同
input = torch.randn(1, 50)  # 生成一个随机输入数据
print('输入:', input)

with torch.no_grad():  # 在评估模式下，不计算梯度
    output = model(input)  # 前向传播，得到输出
print('输出:', output)

output_prob = torch.softmax(output, dim=1)  # 将输出转换为概率分布
print('输出概率分布:', output_prob)

输入: tensor([[ 0.3374, -0.1778, -0.3035, -0.5880,  0.3486,  0.6603, -0.2196, -0.3792,
          0.7671, -1.1925,  0.6984, -1.4097,  0.1794,  1.8951,  0.4954,  0.2692,
         -0.0770, -1.0205, -0.1690,  0.9178,  1.5810,  1.3010,  1.2753, -0.2010,
          0.4965, -1.5723,  0.9666, -1.1481, -1.1589,  0.3255, -0.6315, -2.8400,
         -1.3250,  0.1784, -1.4779,  1.1331, -1.2203,  1.3139,  1.0533,  0.1388,
         -0.2044, -2.2685, -0.2808,  0.7697, -0.6596, -0.7979,  0.1838,  0.2293,
          0.6177, -0.2876]])
输出: tensor([[ 0.1373,  0.0283, -0.2047]])
输出概率分布: tensor([[0.3836, 0.3440, 0.2725]])


# 训练模型

In [13]:
# 训练数据
X_train = torch.tensor([
    [-1.2, 3.1],
    [-0.9, 2.9],
    [-0.5, 2.6],
    [2.3, -1.1],
    [2.7, -1.5],
], dtype=torch.float32)  # 5个样本，每个样本有2个特征
y_train = torch.tensor([0, 0, 0, 1, 1], dtype=torch.long)  # 5个标签

print('训练数据X_train:', X_train)
print('训练数据y_train:', y_train)
print('训练数据X_train的形状:', X_train.shape)
print('训练数据y_train的形状:', y_train.shape)

# 测试数据
X_test = torch.tensor([
    [-0.8, 2.8],
    [2.6, -1.6],
], dtype=torch.float32)  # 2个样本，每个样本有2个特征
y_test = torch.tensor([0, 1], dtype=torch.long)  # 2个标签

print('测试数据X_test:', X_test)
print('测试数据y_test:', y_test)
print('测试数据X_test的形状:', X_test.shape)
print('测试数据y_test的形状:', y_test.shape)

训练数据X_train: tensor([[-1.2000,  3.1000],
        [-0.9000,  2.9000],
        [-0.5000,  2.6000],
        [ 2.3000, -1.1000],
        [ 2.7000, -1.5000]])
训练数据y_train: tensor([0, 0, 0, 1, 1])
训练数据X_train的形状: torch.Size([5, 2])
训练数据y_train的形状: torch.Size([5])
测试数据X_test: tensor([[-0.8000,  2.8000],
        [ 2.6000, -1.6000]])
测试数据y_test: tensor([0, 1])
测试数据X_test的形状: torch.Size([2, 2])
测试数据y_test的形状: torch.Size([2])


In [16]:
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(123)  # 设置随机种子，确保每次运行结果相同

# 数据集
class ToyDataset(Dataset):
    def __init__(self, X, y):
        self.features = X
        self.labels = y

    def __len__(self):
        return self.labels.shape[0]

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]
    
# 创建训练数据集和测试数据集
train_ds = ToyDataset(X_train, y_train)
test_ds = ToyDataset(X_test, y_test)

# 创建数据加载器
train_loader = DataLoader(
    dataset=train_ds,
    batch_size=2,    # 每次加载2个样本
    shuffle=True,    # 打乱顺序
    num_workers=0,
    drop_last=True,  # 丢弃最后一个不完整的批次
)
test_loader = DataLoader(
    dataset=test_ds,
    batch_size=2,   # 每次加载2个样本
    shuffle=False,  # 不打乱顺序
    num_workers=0,
)

for idx, (features, labels) in enumerate(train_loader):
    print(f'批次 {idx+1}:', features, labels)

批次 1: tensor([[ 2.3000, -1.1000],
        [-0.9000,  2.9000]]) tensor([1, 0])
批次 2: tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) tensor([0, 0])


In [23]:
import torch.nn.functional as F

torch.manual_seed(123)  # 设置随机种子，确保每次运行结果相同

model = NeuralNetwork(num_inputs=2, num_outputs=2)  # 创建一个新的模型实例

optimizer = torch.optim.SGD(model.parameters(), lr=0.2)  # 定义优化器，使用随机梯度下降

num_epochs = 10  # 训练的轮数
for epoch in range(num_epochs):
    for batch_idx, (features, labels) in enumerate(train_loader):
        optimizer.zero_grad()  # 清除之前的梯度

        logits = model(features)  # 前向传播，得到输出层的线性输出
        loss = F.cross_entropy(logits, labels)  # 计算交叉熵损失

        loss.backward()  # 反向传播，计算梯度
        optimizer.step()  # 更新模型参数

        print(f'轮次 {epoch+1:03d}/{num_epochs:03d} | 批次 {batch_idx+1:03d}/{len(train_loader):03d} | 损失: {loss.item():.4f}')

    # print(f'第 {epoch+1} 轮训练完成，损失: {loss.item():.4f}')

轮次 001/010 | 批次 001/002 | 损失: 0.8090
轮次 001/010 | 批次 002/002 | 损失: 0.5199
轮次 002/010 | 批次 001/002 | 损失: 0.5626
轮次 002/010 | 批次 002/002 | 损失: 0.3254
轮次 003/010 | 批次 001/002 | 损失: 0.1803
轮次 003/010 | 批次 002/002 | 损失: 0.0773
轮次 004/010 | 批次 001/002 | 损失: 0.1296
轮次 004/010 | 批次 002/002 | 损失: 0.1013
轮次 005/010 | 批次 001/002 | 损失: 0.0150
轮次 005/010 | 批次 002/002 | 损失: 0.1179
轮次 006/010 | 批次 001/002 | 损失: 0.0656
轮次 006/010 | 批次 002/002 | 损失: 0.0191
轮次 007/010 | 批次 001/002 | 损失: 0.0084
轮次 007/010 | 批次 002/002 | 损失: 0.0424
轮次 008/010 | 批次 001/002 | 损失: 0.0211
轮次 008/010 | 批次 002/002 | 损失: 0.0059
轮次 009/010 | 批次 001/002 | 损失: 0.0131
轮次 009/010 | 批次 002/002 | 损失: 0.0093
轮次 010/010 | 批次 001/002 | 损失: 0.0248
轮次 010/010 | 批次 002/002 | 损失: 0.0079


In [26]:

torch.set_printoptions(
    sci_mode=False, 
    precision=4,
)

# 评估模型
model.eval()
with torch.no_grad():
    outputs = model(X_test)
print('测试集上的输出:', outputs)

predicted = torch.argmax(outputs, dim=1)  # 获取预测的类别
print('测试集上的预测类别:', predicted)

accuracy = (predicted == y_test).float().mean()  # 计算准确率
print(f'测试集上的准确率: {accuracy.item()}')

测试集上的输出: tensor([[ 2.1201, -2.9860],
        [-2.0352,  2.2479]])
测试集上的预测类别: tensor([0, 1])
测试集上的准确率: 1.0


# 保存和加载模型

In [27]:
path = './torch-神经网络.pth'

# 保存模型参数到文件
torch.save(model.state_dict(), path)
print(f'模型参数已保存到: {path}')

# 加载模型参数
model = NeuralNetwork(num_inputs=2, num_outputs=2)  # 创建一个新的模型实例
model.load_state_dict(torch.load(path))
print('加载后的模型:', model)
print('加载后的模型参数:', model.state_dict())

模型参数已保存到: ./torch-神经网络.pth
加载后的模型: NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=2, out_features=30, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): GELU(approximate='none')
    (4): Linear(in_features=20, out_features=2, bias=True)
  )
)
加载后的模型参数: OrderedDict({'layers.0.weight': tensor([[-0.3071,  0.0840],
        [-0.3651,  0.3164],
        [-0.6084,  0.5396],
        [-0.5164, -0.5601],
        [-0.4498,  0.3429],
        [-0.2753,  0.3234],
        [-0.6098, -0.4219],
        [-0.2525, -0.1500],
        [-0.5480,  0.4877],
        [-0.1840,  0.2782],
        [ 0.5666,  0.1092],
        [ 0.1453,  0.7284],
        [-0.3288,  0.2643],
        [-0.3172,  0.5311],
        [ 0.6125, -0.6734],
        [ 0.6023,  0.3520],
        [ 0.2966,  0.3149],
        [ 0.7067, -0.1461],
        [-0.6041, -0.1825],
        [-0.5132,  0.1024],
        [-0.0775, -0.4099],
        [-0.0733,  0.7263],
        [-0.0264